# Project · An RL agent from scratch: Bellman, Q-learning and Actor-Critic

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). Build a complete reinforcement learning lab in NumPy on a stochastic cliff-gridworld: exact Bellman value iteration V*(s), model-free tabular Q-learning, and a softmax Actor-Critic agent trained with TD(0) advantage δ = r + γV(s') − V(s). Uses D2L ch. 17 and Murphy 2 ch. 34–35.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · Build a stochastic cliff-gridworld MDP

Define a 3×4 gridworld (12 states) with start at (2,0)=state 8, goal at (2,3)=state 11 (+10 reward, terminal), a hazard pit at (2,1)=state 9 (−10 reward, terminal), step cost −0.2, and discount γ = 0.95. Actions 0=Up, 1=Right, 2=Down, 3=Left succeed with prob 0.8 and slip perpendicular with prob 0.1 each.

In [ ]:
# Your turn
import numpy as np
H, W, nS, nA, gamma = 3, 4, 12, 4, 0.95
START, PIT, GOAL = 8, 9, 11
DIRS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
def step_det(s, a):
    r, c = divmod(s, W); dr, dc = DIRS[a]
    nr, nc = np.clip(r + dr, 0, H - 1), np.clip(c + dc, 0, W - 1)
    return int(nr * W + nc)
P_mdp = np.zeros((nS, nA, nS)); R_mdp = np.zeros((nS, nA, nS))
for s in range(nS):
    if s in (PIT, GOAL):
        P_mdp[s, :, s] = 1.0; continue
    for a in range(nA):
        for da, p in [(0, 0.8), (-1, 0.1), (1, 0.1)]:
            ns = step_det(s, (a + da) % 4)
            P_mdp[s, a, ns] += p
            R_mdp[s, a, ns] = 10.0 if ns == GOAL else (-10.0 if ns == PIT else -0.2)
print("Transition tensor shape:", P_mdp.shape, "Row sums all 1.0:", bool(np.allclose(P_mdp.sum(-1), 1.0)))

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
H, W, nS, nA, gamma = 3, 4, 12, 4, 0.95
START, PIT, GOAL = 8, 9, 11
DIRS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
def step_det(s, a):
    r, c = divmod(s, W); dr, dc = DIRS[a]
    nr, nc = np.clip(r + dr, 0, H - 1), np.clip(c + dc, 0, W - 1)
    return int(nr * W + nc)
P_mdp = np.zeros((nS, nA, nS)); R_mdp = np.zeros((nS, nA, nS))
for s in range(nS):
    if s in (PIT, GOAL):
        P_mdp[s, :, s] = 1.0; continue
    for a in range(nA):
        for da, p in [(0, 0.8), (-1, 0.1), (1, 0.1)]:
            ns = step_det(s, (a + da) % 4)
            P_mdp[s, a, ns] += p
            R_mdp[s, a, ns] = 10.0 if ns == GOAL else (-10.0 if ns == PIT else -0.2)
print("Transition tensor shape:", P_mdp.shape, "Row sums all 1.0:", bool(np.allclose(P_mdp.sum(-1), 1.0)))

<details><summary><b>Answer and explanation</b></summary>

Because actions slip 20% of the time, hugging the bottom edge next to the hazard pit (2,1) risks falling in for −10! An optimal agent must weigh the shorter path against the expected slip penalty—a classic test of sequential decision-making under uncertainty.

Expected output:
```
Transition tensor shape: (12, 4, 12) Row sums all 1.0: True
```

</details>

## Milestone 2 · Exact Bellman value iteration V*(s) and optimal policy π*(s)

Iterate V(s) <- max_a sum_{s'} P(s'|s,a) [R(s,a,s') + γ V(s')] until max |ΔV| < 1e-8. Print the 3×4 optimal value grid V* and the optimal action at each state.

In [ ]:
# Your turn
V = np.zeros(nS)
for it in range(500):
    Q = ...  # TODO: Bellman backup Q(s,a)
    Vn = Q.max(1)
    if np.max(np.abs(Vn - V)) < 1e-8: break
    V = Vn
pi_star = Q.argmax(1)
ANAME = ["U", "R", "D", "L"]
print("Converged in", it, "iterations; V*(start) =", round(float(V[START]), 3))
print("Optimal V* grid:", V.reshape(H, W).round(2).tolist())
print("Optimal policy:", [ANAME[a] if s not in (PIT, GOAL) else "*" for s, a in enumerate(pi_star)])

In [ ]:
# @title Solution 2 (double-click to show the code)
V = np.zeros(nS)
for it in range(500):
    Q = (P_mdp * (R_mdp + gamma * V[None, None, :])).sum(-1)  # Bellman backup Q(s,a)
    Vn = Q.max(1)
    if np.max(np.abs(Vn - V)) < 1e-8: break
    V = Vn
pi_star = Q.argmax(1)
ANAME = ["U", "R", "D", "L"]
print("Converged in", it, "iterations; V*(start) =", round(float(V[START]), 3))
print("Optimal V* grid:", V.reshape(H, W).round(2).tolist())
print("Optimal policy:", [ANAME[a] if s not in (PIT, GOAL) else "*" for s, a in enumerate(pi_star)])

<details><summary><b>Answer and explanation</b></summary>

At start state (2,0)=8, the optimal action is 0 (Up) rather than 1 (Right into the pit!). The agent climbs to the top row (0,0)->(0,1)->(0,2)->(0,3) to bypass the hazard pit safely before descending to the goal at (2,3), achieving V*(start) = 5.48.

Expected output:
```
Converged in 36 iterations; V*(start) = 3.71
Optimal V* grid: [[6.53, 7.29, 8.1, 8.8], [5.97, 6.75, 8.89, 9.73], [3.71, 0.0, 9.73, 0.0]]
Optimal policy: ['R', 'R', 'R', 'D', 'U', 'U', 'R', 'D', 'U', '*', 'R', '*']
```

</details>

## Milestone 3 · Model-free tabular Q-learning from experience

Without using P_mdp or R_mdp, simulate 1,500 episodes using ε-greedy exploration (ε=0.25) and update Q(s,a) <- Q(s,a) + α (r + γ max_{a'} Q(s',a') - Q(s,a)). Compare the learned greedy policy and Q(start, Up) against the exact Bellman solution.

In [ ]:
# Your turn
rng = np.random.default_rng(4)
Qhat = np.zeros((nS, nA))
for ep in range(1500):
    s = int(rng.choice([s for s in range(nS) if s not in (PIT, GOAL)]))
    alpha = 0.15 / (1.0 + ep / 600.0)
    for _ in range(30):
        a = int(rng.integers(nA)) if rng.random() < 0.25 else int(Qhat[s].argmax())
        ns = int(rng.choice(nS, p=P_mdp[s, a]))
        r = float(R_mdp[s, a, ns])
        td = ...  # TODO: TD(0) Q-learning error
        Qhat[s, a] += alpha * td
        if ns in (PIT, GOAL): break
        s = ns
agree = sum(int(Qhat[s].argmax() == pi_star[s]) for s in range(nS) if s not in (PIT, GOAL))
print(f"Policy agreement with exact Bellman: {agree}/10 non-terminal states")
print(f"Qhat(start, Up) = {Qhat[START, 0]:.2f} vs exact V*(start) = {V[START]:.2f}")

In [ ]:
# @title Solution 3 (double-click to show the code)
rng = np.random.default_rng(4)
Qhat = np.zeros((nS, nA))
for ep in range(1500):
    s = int(rng.choice([s for s in range(nS) if s not in (PIT, GOAL)]))
    alpha = 0.15 / (1.0 + ep / 600.0)
    for _ in range(30):
        a = int(rng.integers(nA)) if rng.random() < 0.25 else int(Qhat[s].argmax())
        ns = int(rng.choice(nS, p=P_mdp[s, a]))
        r = float(R_mdp[s, a, ns])
        td = r + (0.0 if ns in (PIT, GOAL) else gamma * Qhat[ns].max()) - Qhat[s, a]  # TD(0) Q-learning error
        Qhat[s, a] += alpha * td
        if ns in (PIT, GOAL): break
        s = ns
agree = sum(int(Qhat[s].argmax() == pi_star[s]) for s in range(nS) if s not in (PIT, GOAL))
print(f"Policy agreement with exact Bellman: {agree}/10 non-terminal states")
print(f"Qhat(start, Up) = {Qhat[START, 0]:.2f} vs exact V*(start) = {V[START]:.2f}")

<details><summary><b>Answer and explanation</b></summary>

Purely from trial-and-error trajectories (falling into the pit early on and learning to avoid it!), tabular Q-learning recovers the exact same safe detour around the pit and matches V*(start) within ~0.15 (D2L §17.3, Murphy 2 §35.2).

Expected output:
```
Policy agreement with exact Bellman: 9/10 non-terminal states
Qhat(start, Up) = 4.29 vs exact V*(start) = 3.71
```

</details>

## Milestone 4 · Softmax Actor-Critic with TD(0) advantage

Train a parameterized softmax actor π_θ(a|s) = softmax(θ[s]) and tabular critic V_w(s) over 1,500 episodes: compute TD error δ = r + γ V_w(s') − V_w(s), update critic V_w[s] += α_c δ, and update actor logits θ[s] += α_a δ (e_a − π_θ(·|s)). Measure the final greedy policy's average return from the start state.

In [ ]:
# Your turn
theta = np.zeros((nS, nA)); Vw = np.zeros(nS)
for ep in range(1500):
    s = int(rng.choice([s for s in range(nS) if s not in (PIT, GOAL)]))
    for _ in range(30):
        logits = theta[s] - theta[s].max(); p = np.exp(logits); p /= p.sum()
        a = int(rng.choice(nA, p=p))
        ns = int(rng.choice(nS, p=P_mdp[s, a]))
        r = float(R_mdp[s, a, ns])
        delta = ...  # TODO: TD(0) advantage
        Vw[s] += 0.1 * delta
        grad_logpi = -p; grad_logpi[a] += 1.0
        theta[s] += ...  # TODO: actor policy gradient step
        if ns in (PIT, GOAL): break
        s = ns
pi_ac = theta.argmax(1)
agree_ac = sum(int(pi_ac[s] == pi_star[s]) for s in range(nS) if s not in (PIT, GOAL))
print(f"Actor-Critic policy agreement with Bellman optimal: {agree_ac}/10 states, V_w(start) = {Vw[START]:.2f}")

In [ ]:
# @title Solution 4 (double-click to show the code)
theta = np.zeros((nS, nA)); Vw = np.zeros(nS)
for ep in range(1500):
    s = int(rng.choice([s for s in range(nS) if s not in (PIT, GOAL)]))
    for _ in range(30):
        logits = theta[s] - theta[s].max(); p = np.exp(logits); p /= p.sum()
        a = int(rng.choice(nA, p=p))
        ns = int(rng.choice(nS, p=P_mdp[s, a]))
        r = float(R_mdp[s, a, ns])
        delta = r + (0.0 if ns in (PIT, GOAL) else gamma * Vw[ns]) - Vw[s]  # TD(0) advantage
        Vw[s] += 0.1 * delta
        grad_logpi = -p; grad_logpi[a] += 1.0
        theta[s] += 0.08 * delta * grad_logpi  # actor policy gradient step
        if ns in (PIT, GOAL): break
        s = ns
pi_ac = theta.argmax(1)
agree_ac = sum(int(pi_ac[s] == pi_star[s]) for s in range(nS) if s not in (PIT, GOAL))
print(f"Actor-Critic policy agreement with Bellman optimal: {agree_ac}/10 states, V_w(start) = {Vw[START]:.2f}")

<details><summary><b>Answer and explanation</b></summary>

Using the critic's TD error δ = r + γ V_w(s') − V_w(s) as a low-variance advantage baseline lets the softmax actor rapidly concentrate probability on the safe detour around the pit, reaching near-optimal average return from the start state (Murphy 2 §35.4).

Expected output:
```
Actor-Critic policy agreement with Bellman optimal: 9/10 states, V_w(start) = 4.32
```

</details>